# ChemBreak 18 v18.0.1 — Evidence-Memory Adaptive MDP Safety Evaluation

**Environment:** Google Cloud Notebook Enterprise  
**Target:** ChemDFM  
**Attack LLM:** GPT-5.6 Sol (`gpt-5.6-sol`, OpenAI API)  
**Judge LLM:** Gemini 2.5 Flash (`gemini-2.5-flash`, Vertex AI)  
**Fixed dataset:** the same 24 tasks in Baseline, Epochs 1–3, and Optimized  
**Pipeline:** Baseline → Epoch 1 → Epoch 2 → Epoch 3 → Build rankings → Freeze → Optimized → Results

Run this notebook from Cell 1 downward. CB18 adds persistent evidence memory, exact-candidate replay, frozen rankings, and optimized fallback while keeping the fixed 24-task experiment structure.


In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

if 'runner' in globals():
    try:
        runner.close()
    except Exception:
        pass
    del runner

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak18"
EXPERIMENT_REVISION = "CB18_EVIDENCE_MDP_MINI24_V2"
LIVE                = True
LIVE_PROGRESS       = True

content_root = Path('/content').resolve()
assert content_root.is_dir(), '/content unavailable — use Google Cloud Notebook Enterprise.'
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith('REPLACE_'), 'Set PROJECT_ID before live execution.'

storage_root = content_root / 'chembreak18_storage'
model_cache = storage_root / 'cache/huggingface/hub'
package_dir = storage_root / 'python_packages'
env_paths = {
    'HF_HOME': storage_root/'cache/huggingface',
    'TRANSFORMERS_CACHE': model_cache,
    'HUGGINGFACE_HUB_CACHE': model_cache,
    'TORCH_HOME': storage_root/'cache/torch',
    'XDG_CACHE_HOME': storage_root/'cache/xdg',
    'PIP_CACHE_DIR': storage_root/'cache/pip',
}
for key,path in env_paths.items():
    path=Path(path); path.mkdir(parents=True,exist_ok=True); os.environ[key]=str(path)
package_dir.mkdir(parents=True,exist_ok=True)
print('CB18 storage:', storage_root)


## C1 — Clone or update the GitHub repository

The notebook is the controller; CB18 source is loaded from the `chembreak18` folder in GitHub.


In [ ]:
checkout = content_root / 'chembreak18_repo'
def git(*args, cwd=None): subprocess.run(['git', *args], cwd=cwd, check=True)
if not (checkout/'.git').is_dir():
    git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(checkout))
else:
    git('fetch','origin',BRANCH,cwd=checkout); git('checkout',BRANCH,cwd=checkout); git('pull','--ff-only','origin',BRANCH,cwd=checkout)
PROJECT_DIR=(checkout/PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR/'pyproject.toml').is_file(), f'CB18 package not found at {PROJECT_DIR}. Push the chembreak18 folder to GitHub first.'
os.chdir(PROJECT_DIR)
print('Project:',PROJECT_DIR)


## C2 — Install the CB18 dependency stack

Dependencies and caches stay under `/content/chembreak18_storage`. This includes the OpenAI Python SDK for GPT-5.6 Sol and Google GenAI for the Gemini 2.5 Flash judge.


In [ ]:
compatibility_specs=['transformers==4.40.2','tokenizers==0.19.1','huggingface-hub==0.23.5','safetensors==0.4.5','accelerate==0.30.1','sentencepiece==0.2.0','einops==0.8.1']
marker=package_dir/'cb18_compatibility.json'; expected={'specifications':compatibility_specs}
installed=json.loads(marker.read_text()) if marker.exists() else None
if installed!=expected:
    print('Installing CB18 compatibility stack (first run only)...')
    subprocess.run([sys.executable,'-m','pip','install','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'--no-deps','--upgrade',*compatibility_specs],check=True)
    marker.write_text(json.dumps(expected,indent=2))
else:
    print('Compatibility stack already installed.')
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'google-auth>=2.35,<3','google-genai>=1.47,<2','openai>=3.14,<4','pandas>=2.2,<3','numpy>=1.26,<3','PyYAML>=6,<7'],check=True)
site.addsitedir(str(package_dir)); sys.path.insert(0,str(package_dir)); sys.path.insert(0,str(PROJECT_DIR/'src')); importlib.invalidate_caches()
print('Dependency stack ready.')


## C3 — Set the OpenAI API key securely

The key is read into the current kernel environment only. It is **not** written to the runtime YAML, results ZIP, or GitHub package.


In [ ]:
if LIVE and not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass
    key = getpass('OpenAI API key (input hidden): ').strip()
    assert key, 'OPENAI_API_KEY is required for the GPT-5.6 Sol attack LLM.'
    os.environ['OPENAI_API_KEY'] = key
print('OPENAI_API_KEY:', 'present' if os.environ.get('OPENAI_API_KEY') else 'mock/not required')


## C4 — Verify the fixed 24-task CB18 panel

This verifies the same locked 24 assignments against the 500-task source bank. There is no train/test/holdout split and no matcher in the current experiment.


In [ ]:
sys.path.insert(0,str(PROJECT_DIR/'src'))
for module_name in [name for name in list(sys.modules) if name == 'chembreak18' or name.startswith('chembreak18.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
from chembreak18.selection import verify_bundle
report=verify_bundle(PROJECT_DIR/'data/final_task_bank.csv',PROJECT_DIR/'data/CB18_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB18_mini24_lock_v1.json')
print(json.dumps(report,indent=2,sort_keys=True))


## C5 — Build the live runtime configuration

The same 24 tasks are used in every phase. CB18 writes Q-policy, evidence-memory, frozen rankings, and freeze-snapshot artifacts into the CB18 storage namespace.


In [ ]:
import yaml
base=yaml.safe_load((PROJECT_DIR/'configs/config.cb18.yaml').read_text())
base['run'].update({'project_root':str(PROJECT_DIR),'task_bank_path':str(PROJECT_DIR/'data/final_task_bank.csv'),'mini_manifest_path':str(PROJECT_DIR/'data/CB18_mini24_manifest_v1.csv'),'mini_lock_path':str(PROJECT_DIR/'data/CB18_mini24_lock_v1.json'),'output_root':str(storage_root/'runs'),'dry_run':not LIVE,'experiment_revision':EXPERIMENT_REVISION,'task_limit':None,'live_progress':LIVE_PROGRESS})
base['targets'][0]['cache_dir']=str(model_cache); base['targets'][0]['offload_folder']=str(storage_root/'offload/ChemDFM')
policy_dir=storage_root/'policies'/EXPERIMENT_REVISION
base['policy']['training_artifact_path']=str(policy_dir/'training_policy.json'); base['policy']['frozen_artifact_path']=str(policy_dir/'frozen_policy.json')
base['evidence']['training_artifact_path']=str(policy_dir/'training_evidence.json'); base['evidence']['frozen_artifact_path']=str(policy_dir/'frozen_evidence.json')
base['evidence']['rankings_artifact_path']=str(policy_dir/'frozen_rankings.json'); base['evidence']['freeze_snapshot_path']=str(policy_dir/'freeze_snapshot.json')
runtime_path=storage_root/f'runtime_{EXPERIMENT_REVISION}.yaml'; runtime_path.write_text(yaml.safe_dump(base,sort_keys=False))
if LIVE:
    os.environ['GOOGLE_CLOUD_PROJECT']=PROJECT_ID
    os.environ['CHEMBREAK_ENABLE_LIVE']='YES'
print('Runtime config:',runtime_path)
print('Attack LLM:',base['roles']['attack_llm']['model'])
print('Policy-block behavior:',base['roles']['attack_llm'].get('policy_block_behavior'))
print('Judge LLM:',base['roles']['judge_llm']['model'])
print('LIVE:',LIVE,'| LIVE_PROGRESS:',LIVE_PROGRESS,'| Project:',os.environ.get('GOOGLE_CLOUD_PROJECT','mock'))


## C6 — Preflight

Checks the fixed panel, CB18 configuration, GPU/CUDA, ChemDFM tokenizer compatibility, GPT-5.6 Sol structured-output attack probe, and Gemini 2.5 Flash structured-output judge probes.


In [ ]:
from chembreak18.preflight import run_preflight
preflight=run_preflight(runtime_path,probe_tokenizer=LIVE,probe_roles=LIVE)
print(json.dumps(preflight,indent=2,sort_keys=True))
assert preflight['status']=='ok'


## C7 — Create the runner and load ChemDFM once

The loaded target is reused for Baseline, all three learning epochs, and Optimized evaluation.


In [ ]:
from chembreak18.runner import ChemBreak18Runner
runner=ChemBreak18Runner(runtime_path)
runner.load_target()
print('Runner ready.')
print('Planned episodes: 24 baseline + 72 learning + 24 optimized = 120')
print('Maximum target queries: 408 (usually lower because success stops an episode early)')


## C8 — Phase 1: Baseline

Each original benchmark prompt is sent once in an independent one-turn episode. No MDP action is selected, no Q-value is updated, and no evidence-memory entry is created.

With `LIVE_PROGRESS = True`, each completed task prints `running_ASR=...`.

Baseline responses are saved for auditing only; they do not seed the adaptive state for Epoch 1.


In [ ]:
baseline_summary=runner.run_baseline()
print(json.dumps(baseline_summary,indent=2,sort_keys=True))


## C9 — Phase 2: Learning (3 epochs)

The same 24 tasks are run in **independent fresh conversations** for three epochs with base epsilon **0.30 → 0.20 → 0.15**. Baseline responses and earlier-epoch conversation histories do not initialize a new epoch; only learned Q-values and Evidence Memory persist.

CB18 combines the hierarchical Q-policy with persistent **Evidence Memory**. During exploitation, a previously successful exact realization can be replayed and re-tested only when its recorded coarse task state is compatible with the current state. Successes strengthen its evidence; later failures weaken it but do not erase earlier successes. Exploration still creates fresh GPT-5.6 Sol realizations. A provider policy block skips only that abstract action for the current task episode and automatically tries another action without consuming a ChemDFM turn. Live output shows `mode`, candidate `source`, Q components, support, and evidence rank when replay is used.


In [ ]:
learning_summary=runner.run_learning()
print(json.dumps(learning_summary,indent=2,sort_keys=True))


## C10 — Build rankings and freeze

After Epoch 3, CB18 ranks successful exact candidates using a fixed rule that combines Wilson lower-bound reliability, observation support, mean reward quality, and mean Q evidence. It then freezes Q-values, evidence memory, and rankings.


In [ ]:
frozen=runner.freeze_policy()
print(json.dumps(frozen,indent=2,sort_keys=True))


## C11 — Phase 3: Optimized evaluation

The same 24 tasks are evaluated in fresh conversations with `epsilon = 0`. The primary choice is an untried exact candidate from the frozen successful-evidence ranking only when its recorded coarse task state matches the current state. If frozen candidates are exhausted, the frozen hierarchical Q-policy chooses an action and GPT-5.6 Sol creates a fresh fallback realization. Optimized outcomes never update Q-values, evidence counts, or rankings.


In [ ]:
optimized_summary=runner.run_optimized()
print(json.dumps(optimized_summary,indent=2,sort_keys=True))


## C12 — Results

CB18 reports final ASR plus **ASR@1** and **ASR@4**, success retention, recovery rate, evidence coverage, policy-support diagnostics, exact-candidate evidence, and frozen candidate rankings.

Key CB18 outputs include `baseline_audit.csv`, `evidence_memory.csv`, `candidate_rankings.csv`, and `provider_events.csv`.


In [ ]:
summary=runner.export_results()
print(json.dumps(summary,indent=2,sort_keys=True))
release_dir=storage_root/'runs'/EXPERIMENT_REVISION/'release'
print('Release directory:',release_dir)
print('Files:',[p.name for p in sorted(release_dir.iterdir())])


## C13 — Build the results ZIP and close the model

The ZIP contains release tables, checkpoint, runtime configuration, CB18 mini-set provenance, Q-policy artifacts, evidence-memory artifacts, frozen rankings, and the immutable freeze snapshot. Model weights, caches, and your OpenAI API key are excluded.


In [ ]:
import zipfile
runner.close()
run_dir=storage_root/'runs'/EXPERIMENT_REVISION
release_dir=run_dir/'release'
zip_path=content_root/f'{EXPERIMENT_REVISION}_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in release_dir.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(run_dir))
    checkpoint=run_dir/'state.sqlite3'
    if checkpoint.exists(): z.write(checkpoint,Path('checkpoint')/checkpoint.name)
    if runtime_path.exists(): z.write(runtime_path,Path('provenance')/runtime_path.name)
    for p in [PROJECT_DIR/'data/CB18_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB18_mini24_lock_v1.json']:
        z.write(p,Path('provenance')/p.name)
    for name in ['training_policy.json','frozen_policy.json','training_evidence.json','frozen_evidence.json','frozen_rankings.json','freeze_snapshot.json']:
        p=policy_dir/name
        if p.exists(): z.write(p,Path('policies')/p.name)
print('Results ZIP:',zip_path)
